In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
# Regression metrics helper
def regression_report(y_true, y_pred, label='', verbose=True):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    if verbose:
        print(f'{label}  RMSE={rmse:,.3f}  MAE={mae:,.3f}  R2={r2:.3f}')
    return rmse, mae, r2


In [3]:
# Load test set (2022, held out, never touched before)
test_df = pd.read_csv('../Data/test.csv')
test_df['Time'] = pd.to_datetime(test_df['Time'], format='mixed', errors='coerce')
print(test_df.shape)


(12934, 19)


In [4]:
# Nowcasting: same 13 features and target used in Comparing_models_nowcasting.ipynb
features = ['GHI', 'temp', 'pressure', 'humidity', 'wind_speed',
            'rain_1h', 'snow_1h', 'clouds_all', 'dayLength',
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
target = 'Power[kW]'

X_test = test_df[features]
y_test = test_df[target]


In [5]:
# Load the final saved nowcasting model and evaluate on test
nowcast_model = joblib.load('nowcast_model.pkl')
test_rmse, test_mae, test_r2 = regression_report(y_test, nowcast_model.predict(X_test), label='RF nowcast test')


RF nowcast test  RMSE=1.903  MAE=1.021  R2=0.859


In [6]:
# Forecasting: rebuild lag and target features on the test set, same as train/val
forecast_df = test_df.sort_values('Time').reset_index(drop=True)

forecast_df['power_lag1'] = forecast_df['Power[kW]'].shift(1)
forecast_df['power_lag2'] = forecast_df['Power[kW]'].shift(2)
forecast_df['power_lag4'] = forecast_df['Power[kW]'].shift(4)
forecast_df['power_roll4'] = forecast_df['Power[kW]'].rolling(window=4).mean()

horizon_steps = {1: '15min', 2: '30min', 3: '45min', 4: '1hour',
                  5: '75min', 6: '90min', 7: '105min', 8: '2hour'}

for steps, label in horizon_steps.items():
    forecast_df[f'target_{label}'] = forecast_df['Power[kW]'].shift(-steps)


In [7]:
# Null out lag/target values that cross a time gap (not exactly 15 min)
gap_back1 = forecast_df['Time'].diff(1)
forecast_df.loc[gap_back1 != pd.Timedelta('15min'), 'power_lag1'] = np.nan

gap_back2 = forecast_df['Time'].diff(2)
forecast_df.loc[gap_back2 != pd.Timedelta('30min'), 'power_lag2'] = np.nan

gap_back4 = forecast_df['Time'].diff(4)
forecast_df.loc[gap_back4 != pd.Timedelta('60min'), 'power_lag4'] = np.nan
forecast_df.loc[gap_back4 != pd.Timedelta('60min'), 'power_roll4'] = np.nan

for steps, label in horizon_steps.items():
    gap_fwd = -forecast_df['Time'].diff(-steps)
    expected_gap = pd.Timedelta(minutes=15 * steps)
    forecast_df.loc[gap_fwd != expected_gap, f'target_{label}'] = np.nan


In [8]:
# Drop rows whose lag features are missing (every horizon model needs these)
lag_cols = ['power_lag1', 'power_lag2', 'power_lag4', 'power_roll4']

rows_before = len(forecast_df)
forecast_df = forecast_df.dropna(subset=lag_cols).reset_index(drop=True)
print('Test rows before:', rows_before, 'after:', len(forecast_df))


Test rows before: 12934 after: 11986


In [9]:
# Weather-only feature list used by the deployed forecasting models
features_weather_only = ['GHI', 'temp', 'pressure', 'humidity', 'wind_speed',
                          'rain_1h', 'snow_1h', 'clouds_all', 'dayLength',
                          'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


In [10]:
# Load each saved horizon model and evaluate on test
test_results = []
for steps, label in horizon_steps.items():
    target_col = f'target_{label}'
    test_h = forecast_df.dropna(subset=[target_col])
    X_test_h = test_h[features_weather_only]
    y_test_h = test_h[target_col]

    model = joblib.load(f'models/forecast_weather_only_{label}.joblib')
    rmse, mae, r2 = regression_report(y_test_h, model.predict(X_test_h), label=label)
    test_results.append({'horizon': label, 'train_rows': None, 'val_rows': len(test_h),
                          'MAE': mae, 'R2': r2, 'RMSE': rmse})

test_results_df = pd.DataFrame(test_results)


15min  RMSE=2.090  MAE=1.184  R2=0.831
30min  RMSE=2.255  MAE=1.319  R2=0.804
45min  RMSE=2.396  MAE=1.436  R2=0.780
1hour  RMSE=2.522  MAE=1.541  R2=0.758
75min  RMSE=2.621  MAE=1.636  R2=0.740
90min  RMSE=2.746  MAE=1.736  R2=0.717
105min  RMSE=2.832  MAE=1.806  R2=0.701
2hour  RMSE=2.921  MAE=1.877  R2=0.684


In [11]:
# Final test results, all horizons
print(test_results_df.round(3))


  horizon train_rows  val_rows    MAE     R2   RMSE
0   15min       None     11749  1.184  0.831  2.090
1   30min       None     11512  1.319  0.804  2.255
2   45min       None     11275  1.436  0.780  2.396
3   1hour       None     11038  1.541  0.758  2.522
4   75min       None     10801  1.636  0.740  2.621
5   90min       None     10564  1.736  0.717  2.746
6  105min       None     10327  1.806  0.701  2.832
7   2hour       None     10090  1.877  0.684  2.921
